# Анализ результатов A/B-тестирования интернет-магазина

- Автор: Топорова Е.В.
- Дата: 14.05.2026

## Введение 
Существует интернет-магазин BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Задачи:
1. Оценить влияние нового интерфейса сайта на поведение пользователей и проверить, увеличилась ли конверсия в покупку в тестовой группе (B) по сравнению с контрольной группой (A) в течение 7 дней после регистрации.
2. Определить, достигнут ли заявленный эффект роста конверсии не менее чем на 3 процентных пункта и является ли результат статистически значимым.

## Описание данных

`ab_test_participants.csv` — таблица участников тестов.  
Структура файла:
- `user_id` — идентификатор пользователя;
- `group` — группа пользователя;
- `ab_test` — название теста;
- `device` — устройство, с которого происходила регистрация.

`ab_test_events.zip` — архив с одним csv-файлом, в котором собраны события 2020 года.  
Структура файла:  
- `user_id` — идентификатор пользователя;
- `event_dt` — дата и время события;
- `event_name` — тип события;
- `details` — дополнительные данные о событии.

## Содержимое проекта

[1. Загрузка данных и знакомство с ними](#1-загрузка-данных)<br>
[2. Оценка корректности проведения теста](#2-оценка-корректности)<br>
[3. Оценка результатов A/B-тестирования](#3-оценка-результатов)

---

<a id="1-загрузка-данных"></a>
## 1 Загрузка данных и знакомство с ними

Загрузим необходимые библиотеки.

In [1]:
import pandas as pd
import seaborn as sns
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

Загрузим данные и изучим их.

Загрузим данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [2]:
df = pd.read_csv('/datasets/yandex_knigi_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/yandex_knigi_data.csv'

Познакомимся с данными и изучим общую информацию о них.

In [ ]:
df.head()

In [ ]:
df.info()

Пропусков в данных нет, типы определены верно. Однако, можно удалить столбец `Unnamed: 0`, так как он дублирует индексы.

In [ ]:
df.drop(columns="Unnamed: 0", inplace=True)
df.head()

In [3]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [4]:
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


In [5]:
participants.head()

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


В датафрейме participants:
- 14 525 строк;
- все столбцы заполнены;
- пропуски отсутствуют;
- типы данных корректны.

In [6]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


In [7]:
events.head()

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


В датафрейме events:
- 787 286 событий;
- пропуски присутствуют только в столбце details;
- столбец event_dt корректно распознан как дата (datetime64).

details - необязательное поле, которое заполняется только для части событий. Такие пропуски можно считать допустимыми.

<a id="2-оценка-корректности"></a>
## 2 Оценка корректности проведения теста
Выделим пользователей, участвующих в тесте.

In [8]:
test_users = participants[participants['ab_test'] == "interface_eu_test"]

Проверим равномерность распределения пользователей по группам теста.

In [9]:
test_users['group'].value_counts(normalize=True)

B    0.503871
A    0.496129
Name: group, dtype: float64

Разница между размерами групп составляет менее 1%, то есть распределение пользователей по группам достаточно равномерно.

Проверим отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [10]:
participants.groupby('user_id')['ab_test'].nunique().value_counts()

1    12751
2      887
Name: ab_test, dtype: int64

Обнаружено 887 пользователей, участвующих одновременно в нескольких тестах. Такие пересечения могут искажать результаты эксперимента, поэтому этих пользователей необходимо исключить из дальнейшего анализа.

In [11]:
cross_users = (
    participants.groupby('user_id')['ab_test']
    .nunique()
)

cross_users = cross_users[cross_users > 1].index

test_users = test_users[
    ~test_users['user_id'].isin(cross_users)
]

Проанализируем данные о пользовательской активности по таблице `ab_test_events`:
- оставим только события, связанные с участвующими в изучаемом тесте пользователями;

In [12]:
test_events = events[events['user_id'].isin(test_users['user_id'])]

- проверим наличие дубликатов среди событий;

In [13]:
sum(test_events.duplicated())

5741

Дубликаты присутствуют, избавимся от них.

In [14]:
test_events.drop_duplicates()

,user_id,event_dt,event_name,details
64672,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0
64946,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8
66585,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32
67873,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48
67930,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0
...,...,...,...,...
777479,F80C9BDDEA02E53C,2020-12-30 10:02:43,purchase,4.49
777488,F80C9BDDEA02E53C,2020-12-30 10:03:51,purchase,4.49
777489,F80C9BDDEA02E53C,2020-12-30 10:03:52,product_cart,NaN
778138,6181F3835EBE66BF,2020-12-30 12:10:39,product_cart,NaN


- определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [15]:
registrations = events[events['event_name'] == 'registration'].groupby(
    'user_id', as_index=False)['event_dt'].min().rename(columns={'event_dt': 'reg_dt'})

test_events = test_events.merge(registrations, on='user_id', how='left')

test_events['lifetime'] = (
    test_events['event_dt'] - test_events['reg_dt']
).dt.days

test_events_7d = test_events[
    (test_events['lifetime'] >= 0) &
    (test_events['lifetime'] <= 6)
]

test_events_7d

,user_id,event_dt,event_name,details,reg_dt,lifetime
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,2020-12-06 14:10:01,0
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8,2020-12-06 14:37:25,0
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32,2020-12-06 17:20:22,0
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48,2020-12-06 19:36:54,0
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0,2020-12-06 19:42:20,0
...,...,...,...,...,...,...
73667,E89AF4EFC757D283,2020-12-29 21:46:43,product_cart,NaN,2020-12-23 09:35:48,6
73670,E89AF4EFC757D283,2020-12-29 21:47:56,product_cart,NaN,2020-12-23 09:35:48,6
73739,A6AFDC94A0D3B23D,2020-12-29 22:47:00,product_page,NaN,2020-12-23 13:53:33,6
73745,A6AFDC94A0D3B23D,2020-12-29 22:48:46,product_page,NaN,2020-12-23 13:53:33,6


Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [16]:
baseline = 0.30
alpha = 0.05
power = 0.80
mde = 0.03

effect = proportion_effectsize(baseline, baseline + mde)

analysis = NormalIndPower()

sample_size = analysis.solve_power(
    effect_size=effect,
    power=power,
    alpha=alpha,
    ratio=1
)

sample_size

3761.596974012117

На каждую группу нужно 3762 пользователя, а на весь тест (A + B) - 7524 пользователя. Обе группы превышают требуемый минимум, следовательно, выборка является достаточной для получения статистически значимых результатов при заданных параметрах.

Рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей, а также конверсию.

In [17]:
buyers = test_events_7d[test_events_7d['event_name'] == 'purchase'] \
    .groupby('user_id', as_index=False) \
    .size() \
    .rename(columns={'size': 'purchase_flag'})

group_sizes = test_users.groupby('group')['user_id'].nunique()

buyers_by_group = test_users[['user_id', 'group']].copy()

buyers_by_group['purchase_flag'] = (
    buyers_by_group['user_id'].isin(buyers['user_id'])
)

buyers_count = buyers_by_group.groupby('group')['purchase_flag'].sum()

conversion = (buyers_count / group_sizes).reset_index()
conversion.columns = ['group', 'conversion']
buyers_count, group_sizes, conversion

(group
 A    1377
 B    1480
 Name: purchase_flag, dtype: int64,
 group
 A    4952
 B    5011
 Name: user_id, dtype: int64,
   group  conversion
 0     A    0.278069
 1     B    0.295350)

Предварительно в тестовой группе наблюдается рост пользовательской активности по сравнению с контрольной.

Конверсия в покупку за первые 7 дней после регистрации в группе A составляет 27.8%, тогда как в группе B - 29.5%. Разница между группами равна примерно 1.7 процентного пункта в пользу тестовой группы.

Таким образом, новый интерфейс приводит к увеличению конверсии, то есть пользователи чаще совершают покупку. Однако рост меньше заявленного в техническом задании порога в 3 процентных пункта, поэтому на данном этапе можно говорить о положительном, но недостаточно сильном эффекте.

<a id="3-оценка-результатов"></a>
## 3 Оценка результатов A/B-тестирования

Проверим изменение конверсии статистическим тестом, учитывая все этапы проверки гипотез.

In [18]:
stat, pval = proportions_ztest(
    [buyers_count['B'], buyers_count['A']],
    [group_sizes['B'], group_sizes['A']],
    alternative='larger'
)

stat, pval

(1.9069651971471773, 0.028262547212292124)

Результаты A/B-теста показывают статистически значимое улучшение конверсии в тестовой группе.

Конверсия в группе B (29.5%) выше, чем в контрольной группе A (27.8%), что даёт прирост около 1.7 процентного пункта. Проведённый z-тест для сравнения долей подтвердил значимость различий (p-value = 0.028), поэтому нулевая гипотеза об отсутствии улучшения отвергается.

Однако наблюдаемый эффект составил лишь 1.7 п.п., что ниже целевого бизнес-порога в 3 п.п.

Новый интерфейс действительно улучшает конверсию пользователей, и эффект статистически подтверждён, однако он недостаточен для выполнения бизнес-цели по величине улучшения.

Важно понять, на каком шаге воронки теряются пользователи, улучшился ли конкретный этап. Это позволит отследить, где интерфейс реально влияет, а где нет. Так как эффект положительный, изменения откатывать не стоит, но нужно развивать эксперимент.